# Making one big dataframe with all speaches in format: # 
 country | ISO3 | session | year | speaker | role | speech_text


**Loading all libraries**

In [176]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)
import pandas as pd
import re # for the patterns

In [177]:
excel = pd.read_excel(os.getenv("xlsx_path"))
print(excel.columns)
# lets drop unnecessary column
speach_df = excel.drop(columns=["Unnamed: 6"])

Index(['Year', 'Session', 'ISO Code', 'Country', 'Name of Person Speaking',
       'Post', 'Unnamed: 6'],
      dtype='str')


# New its time to add speach to each session and person

In [178]:
data_path = os.getenv("data_path")

if not data_path:
    raise ValueError("Environment variable 'data_path' is not set")

# Make the Excel values use the same types as the values in the filenames.
speach_df["ISO Code"] = speach_df["ISO Code"].astype(str).str.strip()
speach_df["Session"] = pd.to_numeric(speach_df["Session"], errors="coerce")
speach_df["Year"] = pd.to_numeric(speach_df["Year"])

if "Speech" not in speach_df.columns:
    speach_df["Speech"] = pd.NA

number_of_sessions = 0
files_processed = 0
files_matched = 0
files_unmatched = 0
files_skipped = 0
skipped_files = []
unmatched_files = []

for folder_name in os.listdir(data_path):
    folder_path = os.path.join(data_path, folder_name)
    if not os.path.isdir(folder_path):
        continue

    for session_name in os.listdir(folder_path):
        session_path = os.path.join(folder_path, session_name)
        if not os.path.isdir(session_path):
            continue

        number_of_sessions += 1

        for txt_file in os.listdir(session_path):
            if not txt_file.endswith(".txt"):
                continue

            parts = txt_file[:-4].split("_")
            if len(parts) != 3:
                files_skipped += 1
                skipped_files.append(txt_file)
                continue

            iso_code, session_num, year = parts
            try:
                session_num = int(session_num)
                year = int(year)
            except ValueError:
                files_skipped += 1
                skipped_files.append(txt_file)
                continue

            try:
                with open(os.path.join(session_path, txt_file), "r", encoding="utf-8") as file:
                    speech_text = file.read()
            except (OSError, UnicodeDecodeError):
                files_skipped += 1
                skipped_files.append(txt_file)
                continue

            files_processed += 1
            mask = (
                (speach_df["ISO Code"] == iso_code.strip())
                & (speach_df["Session"] == session_num)
                & (speach_df["Year"] == year)
            )

            if not mask.any():
                files_unmatched += 1
                unmatched_files.append(txt_file)
                continue

            # Update every row when the metadata contains duplicate keys.
            speach_df.loc[mask, "Speech"] = speech_text
            files_matched += 1

missing_speeches = speach_df["Speech"].isna().sum()
print(f"Sessions found: {number_of_sessions}")
print(f"Speech files processed: {files_processed}")
print(f"Speech files matched: {files_matched}")
print(f"Speech files without a metadata match: {files_unmatched}")
print(f"Files skipped: {files_skipped}")
print(f"Metadata rows without speech: {missing_speeches}")

print("\nSkipped file names:")
print(skipped_files if skipped_files else "None")

print("\nFiles without a metadata match:")
print(unmatched_files if unmatched_files else "None")

Sessions found: 80
Speech files processed: 11141
Speech files matched: 11063
Speech files without a metadata match: 78
Files skipped: 1
Metadata rows without speech: 54

Skipped file names:
['.DS_Store-to-UTF-8.txt']

Files without a metadata match:
['YMD_29_1974.txt', 'YMD_27_1972.txt', 'YMD_34_1979.txt', 'YMD_38_1983.txt', 'IRQ_18_1963.txt', 'SOM_18_1963.txt', 'JAM_18_1963.txt', 'GAB_18_1963.txt', 'UKR_18_1963.txt', 'ZAF_18_1963.txt', 'SDN_18_1963.txt', 'HTI_18_1963.txt', 'SAU_18_1963.txt', 'CUB_18_1963.txt', 'EGY_18_1963.txt', 'SLE_18_1963.txt', 'TZA_18_1963.txt', 'PHL_18_1963.txt', 'POL_18_1963.txt', 'CHN_18_1963.txt', 'ECU_18_1963.txt', 'BFA_18_1963.txt', 'MYS_18_1963.txt', 'CAF_18_1963.txt', 'CYP_18_1963.txt', 'MEX_18_1963.txt', 'SYR_18_1963.txt', 'IND_18_1963.txt', 'MDG_18_1963.txt', 'NOR_18_1963.txt', 'MLI_18_1963.txt', 'DOM_18_1963.txt', 'ETH_18_1963.txt', 'RWA_18_1963.txt', 'BEL_18_1963.txt', 'COD_18_1963.txt', 'CSK_02_1947.txt', 'YMD_35_1980.txt', 'YMD_44_1989.txt', 'YMD_23_

# the number of metadata without speeach is rather small so just drop them

In [179]:
speach_df = speach_df.dropna(subset=["Speech"])

# Instead of nulls in 'Post' column let's add just 'Unknown'

In [180]:
speach_df["Post"] = speach_df["Post"].fillna("Unknown")
speach_df["Country"] = speach_df["Country"].str.strip()

# Fix one corrupted cell (broken Excel formula leaked into 'Country')

In [181]:
speach_df.loc[speach_df["Country"] == "29+C70+A7+D71:E74", "Country"] = "Ethiopia"

## Dataset II - UCDP Conflict Data

Load the UCDP/PRIO conflict dataset.

In [182]:
conflicts_df = pd.read_csv(os.getenv("csv_path"))

## Dataset III - SIPRI Military Spending

Load SIPRI's "Share of GDP" sheet, skipping header/region rows.

In [183]:
def load_sipri_sheet(path, sheet_name, header_row, year_start_col):
    df = pd.read_excel(
        path,
        sheet_name=sheet_name,
        header=header_row,              # row 6 (0-indexed: 5)
        na_values=["..", "xxx", "..."]  # treat these as missing
    )

    # Identify the year columns (everything from year_start_col onward).
    # Doing this BEFORE dropping 'Notes' keeps the column positions predictable.
    year_cols = df.columns[year_start_col:]

    # Drop rows that are region headers (e.g. "Africa", "Asia") or blank spacer rows.
    # These rows always have every year column empty, even though 'Country' itself
    # is populated (so a plain df.dropna(how="all") would NOT catch them, since
    # 'Country' being non-empty makes the row look "not all empty").
    # subset=year_cols restricts the emptiness check to just the year columns,
    # ignoring 'Country'/'Notes' entirely.
    df = df.dropna(subset=year_cols, how="all")

    df = df.drop(columns="Notes")
    return df


load_dotenv()

sipri_path = os.getenv("sipri_path")

# year_start_col=2 because the raw columns are: [Country, Notes, 1949, 1950, ...]
#   index 0 = Country, index 1 = Notes, index 2 = first year (1949)
gdp_share = load_sipri_sheet(sipri_path, "Share of GDP", header_row=5, year_start_col=2)
gdp_share


,Country,1949,1950,1951,1952,1953,1954,1955,1956,1957,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
3,Algeria,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.056527,0.053071,0.049299,0.053270,0.058861,0.048900,0.040575,0.073904,0.079715,0.088306
4,Libya,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.028038,0.035725,NaN,NaN
5,Morocco,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.011807,0.01759,...,0.029821,0.029459,0.029029,0.028658,0.039797,0.037498,0.038049,0.035528,0.034351,0.035431
6,Tunisia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.022250,0.020318,0.019764,0.023833,0.027128,0.026151,0.025512,0.025049,0.024989,0.024950
8,Angola,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.027333,0.025080,0.019558,0.017679,0.018104,0.013932,0.014327,0.014298,0.010339,0.013968
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,Saudi Arabia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.092376,0.094973,0.084159,0.073533,0.084066,0.064310,0.057236,0.063816,0.064052,0.064823
188,Syria,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
189,Türkiye,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.020477,0.020385,0.024921,0.026384,0.021558,0.018145,0.016248,0.016853,0.018403,0.019083
190,United Arab Emirates,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Now, let's melt the table

# Why we melt `gdp_share` into long format

Right now `gdp_share` is **wide**: one row per country, one column per year (1949–2025).
That's fine for a human reading a spreadsheet, but it can't be merged with our other
two datasets, because:

- **Speeches** are one row per `(country, year)`
- **UCDP conflict data** is one row per `(country, year)`

To join SIPRI onto these, it needs to have the same shape: **one row per `(country, year)`**,
not one row per country with 77 year-columns.

**Melting** = unpivoting. It turns each year-column into its own row.

| Before (wide) |      |      |      |
|---|---|---|---|
| Country | 1949 | 1950 | 1951 |
| Algeria | NaN  | NaN  | 2.1  |

| After (long) |      |      |
|---|---|---|
| Country | Year | milex_pct_gdp |
| Algeria | 1949 | NaN |
| Algeria | 1950 | NaN |
| Algeria | 1951 | 2.1 |

Once every row is a single `(country, year, value)`, we can merge it directly onto
the speeches and conflict data using `(Country, Year)` as the shared key.

In [184]:
gdp_long = gdp_share.melt(
    id_vars=["Country"],
    var_name="Year",
    value_name="milex_pct_gdp"
)

gdp_long["Year"] = gdp_long["Year"].astype(int)
gdp_long = gdp_long.dropna(subset=["milex_pct_gdp"])
gdp_df = gdp_long

# Standardize historical/alias country names

Speeches, UCDP, and SIPRI each use different spellings for the same countries
(whitespace, typos, formal names, and old names). We standardize speech names to
match SIPRI's style using a hand-built rename dictionary, built by comparing the
three datasets' unique country lists.

In [185]:
# True 1-to-1 renames: same country/entity, just an old name -> current SIPRI-style name
rename_dict = {
    "Burma": "Myanmar",
    "Union of Burma": "Myanmar",
    "Union of Myanmar": "Myanmar",
    "Myanmar / Burma": "Myanmar",
    "Ceylon": "Sri Lanka",
    "Ceylon (Sri Lanka)": "Sri Lanka",
    "Zaire": "Congo, DR",
    "Congo, Leopoldville": "Congo, DR",
    "Congo, the Democratic Republic of the/Zaire": "Congo, DR",
    "Democratic Republic of Congo": "Congo, DR",
    "Democratic Republic Of Congo": "Congo, DR",
    "Democratic Republic of the Congo": "Congo, DR",
    "DRC": "Congo, DR",
    "Congo, Brazzaville": "Congo, Republic",
    "Congo (Brazzaville": "Congo, Republic",
    "Congo, the Peoples Republic of the": "Congo, Republic",
    "Dahomey": "Benin",
    "Benin (Dahomey)": "Benin",
    "Upper Volta": "Burkina Faso",
    "Upper Volta/Burkina Faso": "Burkina Faso",
    "Siam": "Thailand",
    "Swaziland": "Eswatini",
    "Kingdom of Eswatini": "Eswatini",
    "Macedonia": "North Macedonia",
    "Macedonia, the former Yugoslav Republic of": "North Macedonia",
    "Republic of North Macedonia": "North Macedonia",
    "Kampuchea": "Cambodia",
    "Democratic Kampuchea": "Cambodia",
    "Khmer Republic": "Cambodia",
    "Cambodia /Kampuchea": "Cambodia",
    "Western Samoa": "Samoa",
    "Independent State of Samoa": "Samoa",
    "Samoa": "Samoa",
    "Timor-Leste": "Timor Leste",
    "Democratic Republic of Timor-Leste": "Timor Leste",
    "Czech Republic": "Czechia",
    "Turkey": "Türkiye",
    "Republic of Turkey": "Türkiye",
}

Apply the rename dictionary to speech country names.

In [186]:
speach_df["Country"] = speach_df["Country"].replace(rename_dict)
print(len(speach_df["Country"].unique()))

514


# Flag historical states that split/merged/dissolved (no valid modern equivalent)

USSR, Czechoslovakia, Yugoslavia, divided Germany, and Yemen before 1990 no
longer exist as single modern entities. Mapping them to one successor state
would be factually wrong, so we flag these rows rather than dropping or
force-mapping them. They remain usable for text analysis but are excluded from
models requiring SIPRI/UCDP country-year variables.

In [187]:
impossible_bucket = [
    # USSR variants
    "USSR", "USSR ", "Union of Socialist Soviet Republics",
    "Union of Sovier Socialist Republics", "Union of Soviet Socialist Republics",
    "Russian Federation (USSR)",

    # Czechoslovakia
    "Czechoslovakia", "Czechoslovak Socialist Republic",

    # Yugoslavia
    "Yugoslavia", "Serbia and Montenegro",

    # Germany (pre-unification, both sides)
    "German Democratic Republic", "Germany Democratic Republic",
    "Democratic Republic of Germany", "GDR", "DDR", "FDR", "FDR (Germany)",
    "Federal Republic Germany", "Federal Republic of Germany",
    "Republica Federal Alemana",

    # Yemen (pre-1990 unification, both sides)
    "Yemen Arab Republic",
    # (South Yemen / YMD already exists elsewhere in your speech metadata as "YMD" ISO code,
    #  worth checking if its Country column has its own separate string too)

    # Byelorussian/Ukrainian SSR (distinct UN seats pre-1991, now part of modern Belarus/Ukraine
    # via full independence, not a simple rename - USSR breakup case)
    "BSSR", "BelSSR", "Byelorussian Soviet Socialist Republic",
    "Byelorussian Soviet Socialist Repulic",
    "Belarus, Byelorussian Soviet Socialist Republic",
    "UkrSSR", "Ukraine SSR", "Ukrainian Soviet Socialist Republic",
    "Ukranian SSR", "Ukranian Soviet Socialist Republic",
]

# Flag these rows explicitly instead of dropping them
speach_df["historical_entity_unmatched"] = speach_df["Country"].isin(impossible_bucket)

print(f"Rows flagged as historical/unmatched: {speach_df['historical_entity_unmatched'].sum()}")
print(f"Out of total rows: {len(speach_df)}")

# Which years do these cluster in? (sanity check - should be Cold War era + early 90s)
print(speach_df[speach_df["historical_entity_unmatched"]]["Year"].value_counts().sort_index())

Rows flagged as historical/unmatched: 250
Out of total rows: 11080
Year
1946    5
1947    4
1948    5
1949    5
1950    5
1951    6
1952    5
1953    5
1954    5
1955    5
1956    5
1957    6
1958    7
1959    6
1960    5
1961    5
1962    5
1963    4
1964    5
1965    5
1966    5
1967    5
1968    5
1969    5
1970    1
1971    4
1972    5
1973    5
1974    7
1975    5
1976    8
1977    5
1978    7
1979    5
1980    6
1981    5
1982    6
1983    4
1984    6
1985    8
1986    6
1987    5
1988    7
1989    6
1990    6
1991    2
1992    1
2001    1
2002    1
2003    1
2004    1
2005    1
2019    1
2020    1
Name: count, dtype: int64


"Federal Republic of Germany" means different things before/after 1990 - fix that ambiguity.

In [166]:
# "Federal Republic of Germany" is ambiguous: pre-1990 it is historical West Germany;
# post-1990 it is modern Germany's official long-form name.
mask_germany_post1990 = (
    (speach_df["Country"] == "Federal Republic of Germany")
    & (speach_df["Year"] >= 1990)
)
speach_df.loc[mask_germany_post1990, "Country"] = "Germany"
speach_df.loc[mask_germany_post1990, "historical_entity_unmatched"] = False

Result: 250 of 11,080 speeches (2.3%) flagged as historical/unmatched, concentrated in 1946-1992. Kept for text analysis, excluded from SIPRI/UCDP merges.

Split UCDP's multi-country "location" field into one row per country.

In [188]:
# Split multi-country location entries into separate rows.
conflicts_df = conflicts_df.assign(
    Country=conflicts_df["location"].str.split(", ")
).explode("Country")

conflicts_df["Country"] = conflicts_df["Country"].str.strip()

Apply the same rename + historical-entity flagging used for speeches, adapted to UCDP's naming style.

In [189]:
# UCDP-specific renames: same entities as before, but UCDP phrases them differently.
ucdp_rename_dict = {
    "Cambodia (Kampuchea)": "Cambodia",
    "Myanmar (Burma)": "Myanmar",
    "Madagascar (Malagasy)": "Madagascar",
    "DR Congo (Zaire)": "Congo, DR",
    "Congo": "Congo, Republic",
}

rename_dict.update(ucdp_rename_dict)

ucdp_impossible_bucket = [
    "Russia (Soviet Union)",
    "South Yemen",
    "Yemen (North Yemen)",
    "Serbia (Yugoslavia)",
    "South Vietnam",
    "Vietnam (North Vietnam)",
    "Zimbabwe (Rhodesia)",
]

impossible_bucket.extend(ucdp_impossible_bucket)

conflicts_df["Country"] = conflicts_df["Country"].replace(rename_dict)
conflicts_df["historical_entity_unmatched"] = conflicts_df["Country"].isin(impossible_bucket)

Some UCDP labels stay fixed for a conflict's whole record even after the country's status changed - correct those post-transition rows.

In [190]:
# Russia (Soviet Union): USSR dissolved in 1991. Post-1991 rows are modern Russia.
mask_russia_post1991 = (
    (conflicts_df["Country"] == "Russia (Soviet Union)")
    & (conflicts_df["year"] >= 1992)
)
conflicts_df.loc[mask_russia_post1991, "Country"] = "Russia"
conflicts_df.loc[mask_russia_post1991, "historical_entity_unmatched"] = False

# Vietnam (North Vietnam): unified with South Vietnam in 1976. Post-1976 rows are unified Vietnam.
mask_vietnam_post1976 = (
    (conflicts_df["Country"] == "Vietnam (North Vietnam)")
    & (conflicts_df["year"] >= 1976)
)
conflicts_df.loc[mask_vietnam_post1976, "Country"] = "Viet Nam"
conflicts_df.loc[mask_vietnam_post1976, "historical_entity_unmatched"] = False

# Yemen (North Yemen): unified with South Yemen in 1990. Post-1990 rows are unified Yemen.
mask_yemen_post1990 = (
    (conflicts_df["Country"] == "Yemen (North Yemen)")
    & (conflicts_df["year"] >= 1990)
)
conflicts_df.loc[mask_yemen_post1990, "Country"] = "Yemen"
conflicts_df.loc[mask_yemen_post1990, "historical_entity_unmatched"] = False

Final naming fixes to match SIPRI, plus flagging Hyderabad (not a modern country).

In [191]:
# Final UCDP -> SIPRI naming fixes, plus Hyderabad (a 1948-absorbed princely state,
# not a country in any modern dataset - flagged rather than mapped anywhere).
final_rename_dict = {
    "Bosnia-Herzegovina": "Bosnia and Herzegovina",
    "Gambia": "Gambia, The",
    "Ivory Coast": "Cote d'Ivoire",
    "Kyrgyzstan": "Kyrgyz Republic",
    "South Korea": "Korea, South",
}
rename_dict.update(final_rename_dict)
conflicts_df["Country"] = conflicts_df["Country"].replace(final_rename_dict)

impossible_bucket.append("Hyderabad")
conflicts_df["historical_entity_unmatched"] = conflicts_df["Country"].isin(impossible_bucket)

UCDP cleanup done: 98 of 2,990 rows flagged as historical (time-accurate). North Korea, Comoros, Grenada, and Suriname have no SIPRI military-spending data - a genuine gap, not a naming issue.

Phase 1 (data cleaning) is complete. Next step: perform the final three-way merge on country and year.